# 01 · Data exploration and naive baselines (dataset v1)

Load the quality-controlled state-wise daily *energy met* data (Hunt & Bloomfield, 2013–2024) and ERA5 weather for Delhi; resolve duplicate and conflicting dates; build a clean Delhi series; measure how many 7-day forecast windows survive the gaps; establish weekly-naive and trend-naive baselines; build a leakage-checked lag-feature matrix.

**Key results**
- 3,893 observed days out of 4,136; 243 missing days in 96 blocks (longest: 82 days); 7 conflicting dates dropped.
- Only 52% of candidate 7-day windows are usable — gaps, not models, are the first constraint.
- Chronological splits: development (targets < 2022), validation (2022), test (2023–Apr 2024); boundary-crossing windows removed.
- Weekly-naive MAE: 6.25 MU (dev) · 7.92 (validation) · 8.35 (test). Trend-adjusted naive is worse.

**Outputs:** `data/processed/delhi_energy_clean.csv`, `gridpulse_features_v1.parquet`, `seasonal_naive_predictions.csv`, `trend_naive_predictions.csv`

> Paths are relative to `notebooks/`. Outputs are from the original research run; see [`data/README.md`](../data/README.md) for data that must be downloaded.

➡ Next: [02 · XGBoost on lag features](02_xgboost_baseline.ipynb)

Check the downloaded CSV dataset

In [5]:
import pandas as pd

df = pd.read_csv("../data/raw/NCT_of_Delhi.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst rows:")
print(df.head(3).to_string())

print("\nMissing values (%):")
print((df.isna().mean() * 100).round(2))

print("\nColumn types:")
print(df.dtypes)

Shape: (16554, 22)

Columns: ['date', '10m_u_component_of_wind_min', '10m_u_component_of_wind_mean', '10m_u_component_of_wind_max', '10m_v_component_of_wind_min', '10m_v_component_of_wind_mean', '10m_v_component_of_wind_max', '2m_temperature_min', '2m_temperature_mean', '2m_temperature_max', '2m_dewpoint_temperature_min', '2m_dewpoint_temperature_mean', '2m_dewpoint_temperature_max', 'surface_solar_radiation_downwards_min', 'surface_solar_radiation_downwards_mean', 'surface_solar_radiation_downwards_max', 'total_cloud_cover_min', 'total_cloud_cover_mean', 'total_cloud_cover_max', 'utci_min', 'utci_mean', 'utci_max']

First rows:
         date  10m_u_component_of_wind_min  10m_u_component_of_wind_mean  10m_u_component_of_wind_max  10m_v_component_of_wind_min  10m_v_component_of_wind_mean  10m_v_component_of_wind_max  2m_temperature_min  2m_temperature_mean  2m_temperature_max  2m_dewpoint_temperature_min  2m_dewpoint_temperature_mean  2m_dewpoint_temperature_max  surface_solar_radiation

Inspect Conflicts

In [12]:

from pathlib import Path
import pandas as pd

DATA = Path("../data/raw")

df = pd.read_csv(
    DATA / "corrected_daily_energy_met_MU.csv"
)

print("Columns:", df.columns.tolist())

# Identify date and Delhi columns without assuming
# their exact capitalization.
date_col = next(
    c for c in df.columns
    if c.lower().strip() in ("date", "datetime")
)
delhi_col = next(
    c for c in df.columns
    if c.lower().strip() in ("delhi", "nct_of_delhi")
)

df[date_col] = pd.to_datetime(
    df[date_col], errors="raise"
)
df[delhi_col] = pd.to_numeric(
    df[delhi_col], errors="coerce"
)

# All duplicate-date records, not just the first.
dupes = df[
    df.duplicated(subset=[date_col], keep=False)
].copy()

# Dates where Delhi readings disagree.
conflict_dates = (
    dupes.groupby(date_col)[delhi_col]
    .nunique(dropna=False)
    .loc[lambda s: s > 1]
    .index
)

conflicts = (
    dupes[dupes[date_col].isin(conflict_dates)]
    .sort_values(date_col)
)

print("Total rows:", len(df))
print("Unique dates:", df[date_col].nunique())
print("Duplicate-date groups:", dupes[date_col].nunique())
print("Conflicting Delhi dates:", len(conflict_dates))

# Show entire rows to help identify whether
# the conflict also affects other states.
print(conflicts.to_string(index=False))

conflicts.to_csv(
    "delhi_energy_conflicts.csv", index=False
)


Columns: ['Date', 'Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli', 'Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal', 'AMNSIL', 'Damodar Valley Corporation', 'Total']
Total rows: 3955
Unique dates: 3900
Duplicate-date groups: 55
Conflicting Delhi dates: 7
      Date  Andaman and Nicobar Islands  Andhra Pradesh  Arunachal Pradesh  Assam  Bihar  Chandigarh  Chhattisgarh  Dadra and Nagar Haveli  Daman and Diu  Delhi  Goa  Gujarat  Haryana  Himachal Pradesh  Jammu and Kashmir  Jharkhand  Karnataka  Kerala  Lakshadweep  Madhya Pradesh  Maharashtra  Manipur  Meghalaya  Mizoram  Nagaland

Compare Corrected data with Original data

In [13]:

from pathlib import Path
import pandas as pd

DATA = Path("../data/raw")

corrected = pd.read_csv(
    DATA / "corrected_daily_energy_met_MU.csv",
    parse_dates=["Date"]
)

original = pd.read_csv(
    DATA / "daily_energy_met_MU.csv",
    parse_dates=["Date"]
)

conflict_dates = [
    "2014-04-28",
    "2014-12-10",
    "2015-04-05",
    "2016-05-26",
    "2016-10-01",
    "2017-04-14",
    "2019-02-25",
]

for date_str in conflict_dates:
    date = pd.Timestamp(date_str)

    print(f"\n{'=' * 65}")
    print(f"DATE: {date.date()}")
    print("=" * 65)

    print("\nCORRECTED VALUES:")
    print(
        corrected.loc[
            corrected["Date"] == date,
            ["Date", "Delhi", "Total"]
        ].to_string(index=False)
    )

    print("\nORIGINAL VALUES:")
    print(
        original.loc[
            original["Date"] == date,
            ["Date", "Delhi", "Total"]
        ].to_string(index=False)
    )

    print("\nNEIGHBORING CORRECTED DATES:")
    nearby = corrected.loc[
        corrected["Date"].between(
            date - pd.Timedelta(days=2),
            date + pd.Timedelta(days=2)
        ),
        ["Date", "Delhi", "Total"]
    ].sort_values("Date")

    print(nearby.to_string(index=False))



DATE: 2014-04-28

CORRECTED VALUES:
      Date  Delhi  Total
2014-04-28   84.4 2984.6
2014-04-28   87.3 3023.8

ORIGINAL VALUES:
      Date  Delhi  Total
2014-04-28   84.4 2984.6
2014-04-28   87.3 3023.8

NEIGHBORING CORRECTED DATES:
      Date  Delhi  Total
2014-04-26   78.8 2988.6
2014-04-27   76.3 2918.4
2014-04-28   84.4 2984.6
2014-04-28   87.3 3023.8
2014-04-30   90.7 2992.9

DATE: 2014-12-10

CORRECTED VALUES:
      Date  Delhi  Total
2014-12-10   59.0 2867.8
2014-12-10   59.3 2872.3

ORIGINAL VALUES:
      Date  Delhi  Total
2014-12-10   59.0 2867.8
2014-12-10   59.3 2872.3

NEIGHBORING CORRECTED DATES:
      Date  Delhi  Total
2014-12-08   57.4 2840.9
2014-12-09   59.3 2898.2
2014-12-10   59.0 2867.8
2014-12-10   59.3 2872.3
2014-12-11   54.2 2864.5
2014-12-12   58.7 2821.6
2014-12-12   58.7 2821.6

DATE: 2015-04-05

CORRECTED VALUES:
      Date  Delhi  Total
2015-04-05   69.2 2862.3
2015-04-05   57.8 2786.9

ORIGINAL VALUES:
      Date  Delhi  Total
2015-04-05   69.2 2862.3


Check whether duplicate dates explain missing dates

In [15]:

from pathlib import Path
import pandas as pd

DATA = Path("../data/raw")

df = pd.read_csv(
    DATA / "corrected_daily_energy_met_MU.csv",
    parse_dates=["Date"]
)

dates = set(df["Date"])

duplicate_dates = df.loc[
    df.duplicated("Date", keep=False), "Date"
].drop_duplicates().sort_values()

results = []

for date in duplicate_dates:
    rows = df[df["Date"] == date]
    next_day = date + pd.Timedelta(days=1)

    results.append({
        "duplicate_date": date.date(),
        "row_count": len(rows),
        "next_day_missing": next_day not in dates,
        "delhi_conflict": (
            rows["Delhi"].nunique(dropna=False) > 1
        ),
        "delhi_values": rows["Delhi"].tolist(),
    })

report = pd.DataFrame(results)

print(report.to_string(index=False))

print("\nSUMMARY")
print("Duplicate-date groups:", len(report))
print(
    "Followed by missing day:",
    report["next_day_missing"].sum()
)
print(
    "Conflicting Delhi groups followed by missing day:",
    (
        report["next_day_missing"]
        & report["delhi_conflict"]
    ).sum()
)


duplicate_date  row_count  next_day_missing  delhi_conflict   delhi_values
    2013-06-26          2             False           False [102.2, 102.2]
    2013-07-04          2             False           False [108.2, 108.2]
    2013-12-31          2              True           False   [65.2, 65.2]
    2014-04-05          2             False           False   [63.3, 63.3]
    2014-04-08          2             False           False   [65.1, 65.1]
    2014-04-11          2             False           False   [67.4, 67.4]
    2014-04-21          2             False           False   [70.7, 70.7]
    2014-04-28          2              True            True   [84.4, 87.3]
    2014-07-10          2             False           False [119.3, 119.3]
    2014-12-10          2             False            True   [59.0, 59.3]
    2014-12-12          2             False           False   [58.7, 58.7]
    2014-12-27          2             False           False   [74.7, 74.7]
    2014-12-31          2

Identify whether duplicates are genuinely identical

In [16]:
print("\nFULL-ROW DUPLICATE ANALYSIS")

duplicate_groups = df[
    df.duplicated("Date", keep=False)
].groupby("Date")

identical = []
conflicting = []

for date, group in duplicate_groups:
    values = group.drop(columns=["Date"])

    if len(values.drop_duplicates()) == 1:
        identical.append(date)
    else:
        conflicting.append(date)

print("Completely identical groups:", len(identical))
print("Groups differing in any column:", len(conflicting))
print("Conflicting dates:", conflicting)


FULL-ROW DUPLICATE ANALYSIS
Completely identical groups: 43
Groups differing in any column: 12
Conflicting dates: [Timestamp('2013-06-26 00:00:00'), Timestamp('2013-07-04 00:00:00'), Timestamp('2014-04-21 00:00:00'), Timestamp('2014-04-28 00:00:00'), Timestamp('2014-12-10 00:00:00'), Timestamp('2014-12-27 00:00:00'), Timestamp('2015-01-27 00:00:00'), Timestamp('2015-04-05 00:00:00'), Timestamp('2016-05-26 00:00:00'), Timestamp('2016-10-01 00:00:00'), Timestamp('2017-04-14 00:00:00'), Timestamp('2019-02-25 00:00:00')]


Create a clean Delhi daily series

In [17]:

from pathlib import Path
import pandas as pd

DATA = Path("../data")
OUTPUT = DATA / "processed"
OUTPUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    DATA / "raw/corrected_daily_energy_met_MU.csv",
    parse_dates=["Date"],
    usecols=["Date", "Delhi"],
)

# Count distinct values for every date.
counts = df.groupby("Date")["Delhi"].nunique(dropna=False)
conflict_dates = counts[counts > 1].index

# Preserve source records for unresolved dates.
conflicts = df[df["Date"].isin(conflict_dates)].copy()
conflicts.to_csv(
    OUTPUT / "delhi_energy_conflicts.csv",
    index=False,
)

# Collapse dates with one unique Delhi value.
# Conflicting dates are deliberately set to NaN.
daily = df.groupby("Date")["Delhi"].first().to_frame()

daily.loc[conflict_dates, "Delhi"] = float("nan")

# Create the full calendar, without filling missing values.
calendar = pd.date_range(
    df["Date"].min(),
    df["Date"].max(),
    freq="D",
    name="Date",
)

daily = daily.reindex(calendar)

daily["is_missing"] = daily["Delhi"].isna()
daily["is_conflict"] = daily.index.isin(conflict_dates)

daily = daily.rename(
    columns={"Delhi": "energy_met_MU"}
)

daily.to_csv(OUTPUT / "delhi_energy_clean.csv")

print("Calendar days:", len(daily))
print("Observed days:", daily["energy_met_MU"].notna().sum())
print("Missing calendar/invalid days:", daily["is_missing"].sum())
print("Conflicting dates:", daily["is_conflict"].sum())
print("Date range:", daily.index.min(), daily.index.max())


Calendar days: 4136
Observed days: 3893
Missing calendar/invalid days: 243
Conflicting dates: 7
Date range: 2013-01-01 00:00:00 2024-04-28 00:00:00


Examine missing-date patterns

In [18]:

from pathlib import Path
import pandas as pd

DATA = Path("../data")

df = pd.read_csv(
    DATA / "processed/delhi_energy_clean.csv",
    parse_dates=["Date"],
    index_col="Date",
)

missing = df["energy_met_MU"].isna()

# Identify consecutive blocks of missing days.
groups = missing.ne(missing.shift()).cumsum()

gaps = (
    df.loc[missing]
    .groupby(groups[missing])
    .apply(lambda x: pd.Series({
        "start": x.index.min(),
        "end": x.index.max(),
        "days": len(x),
    }))
    .reset_index(drop=True)
    .sort_values("days", ascending=False)
)

print("Total missing days:", missing.sum())
print("Number of missing blocks:", len(gaps))
print("Longest missing block:", gaps["days"].max())

print("\n10 longest missing blocks:")
print(gaps.head(10).to_string(index=False))

print("\nMissing observations by year:")
print(missing.groupby(df.index.year).sum())

print("\nMissing observations by month:")
print(missing.groupby(df.index.month).sum())


Total missing days: 243
Number of missing blocks: 96
Longest missing block: 82

10 longest missing blocks:
     start        end  days
2013-01-08 2013-03-30    82
2017-02-11 2017-02-19     9
2017-03-17 2017-03-23     7
2014-01-01 2014-01-07     7
2020-09-17 2020-09-23     7
2020-08-24 2020-08-29     6
2018-08-18 2018-08-22     5
2020-09-11 2020-09-14     4
2020-08-02 2020-08-05     4
2021-05-31 2021-06-03     4

Missing observations by year:
Date
2013    94
2014    11
2015    14
2016    16
2017    30
2018     6
2019     4
2020    50
2021    13
2022     4
2023     1
2024     0
Name: energy_met_MU, dtype: int64

Missing observations by month:
Date
1     33
2     43
3     46
4     12
5      6
6     17
7      9
8     21
9     24
10    14
11    11
12     7
Name: energy_met_MU, dtype: int64


Measure eligible forecasting windows

In [19]:

from pathlib import Path
import pandas as pd

df = pd.read_csv(
    DATA / "processed/delhi_energy_clean.csv",
    parse_dates=["Date"],
    index_col="Date"
).sort_index()

y = df["energy_met_MU"]

CONTEXT = 28
HORIZON = 7

records = []

for i in range(CONTEXT, len(y) - HORIZON + 1):
    history = y.iloc[i-CONTEXT:i]
    future = y.iloc[i:i+HORIZON]

    records.append({
        "origin": y.index[i-1],
        "eligible": (
            history.notna().all()
            and future.notna().all()
        ),
        "missing_history": history.isna().sum(),
        "missing_future": future.isna().sum(),
    })

windows = pd.DataFrame(records)
windows["year"] = windows["origin"].dt.year

print("Total candidate windows:", len(windows))
print("Eligible windows:", windows["eligible"].sum())
print(
    "Eligibility rate:",
    f"{windows['eligible'].mean():.2%}"
)

print("\nEligible windows by year:")
print(
    windows.groupby("year")["eligible"].sum()
)

print("\nMissing-history failures:",
      (windows["missing_history"] > 0).sum())
print("Missing-target failures:",
      (windows["missing_future"] > 0).sum())


Total candidate windows: 4102
Eligible windows: 2148
Eligibility rate: 52.36%

Eligible windows by year:
year
2013     69
2014    266
2015    117
2016    110
2017    117
2018    291
2019    259
2020    116
2021    103
2022    258
2023    330
2024    112
Name: eligible, dtype: int64

Missing-history failures: 1741
Missing-target failures: 707


Calculate the seasonal-naive baseline

In [20]:

from pathlib import Path
import numpy as np
import pandas as pd

df = pd.read_csv(
    DATA / "processed/delhi_energy_clean.csv",
    parse_dates=["Date"],
    index_col="Date",
).sort_index()

y = df["energy_met_MU"]

CONTEXT = 28
HORIZON = 7

records = []

for i in range(CONTEXT, len(y) - HORIZON + 1):
    history = y.iloc[i-CONTEXT:i]
    future = y.iloc[i:i+HORIZON]

    if history.isna().any() or future.isna().any():
        continue

    origin = y.index[i - 1]

    # Split by the last forecasted date to prevent
    # development targets entering validation/test.
    target_end = future.index[-1]

    if target_end < pd.Timestamp("2022-01-01"):
        split = "development"
    elif target_end < pd.Timestamp("2023-01-01"):
        split = "validation"
    else:
        split = "test"

    # Forecast next seven days from previous seven days.
    forecast = history.iloc[-7:].to_numpy()
    actual = future.to_numpy()

    for h in range(HORIZON):
        records.append({
            "origin": origin,
            "target_date": future.index[h],
            "horizon": h + 1,
            "actual": actual[h],
            "predicted": forecast[h],
            "split": split,
        })

results = pd.DataFrame(records)
results["error"] = results["actual"] - results["predicted"]
results["abs_error"] = results["error"].abs()
results["squared_error"] = results["error"] ** 2

print("\nSeasonal naive baseline")
for split, part in results.groupby("split"):
    mae = part["abs_error"].mean()
    rmse = np.sqrt(part["squared_error"].mean())
    wape = (
        part["abs_error"].sum()
        / part["actual"].abs().sum()
        * 100
    )
    bias = part["error"].mean()

    print(
        f"{split}: "
        f"MAE={mae:.3f} MU, "
        f"RMSE={rmse:.3f}, "
        f"WAPE={wape:.2f}%, "
        f"Bias={bias:.3f} MU"
    )

OUTPUT = DATA / "processed"
results.to_csv(
    OUTPUT / "seasonal_naive_predictions.csv",
    index=False,
)



Seasonal naive baseline
development: MAE=6.250 MU, RMSE=8.811, WAPE=7.57%, Bias=0.093 MU
test: MAE=8.335 MU, RMSE=11.621, WAPE=9.01%, Bias=0.044 MU
validation: MAE=7.785 MU, RMSE=10.828, WAPE=8.57%, Bias=-0.447 MU


Diagnose baseline errors

In [21]:

from pathlib import Path
import pandas as pd

df = pd.read_csv(
    DATA / "processed/seasonal_naive_predictions.csv",
    parse_dates=["origin", "target_date"],
)

print("\nMAE BY FORECAST HORIZON")
horizon_mae = (
    df.groupby(["split", "horizon"])["abs_error"]
    .mean()
    .unstack("split")
)
print(horizon_mae.round(3).to_string())

df["month"] = df["target_date"].dt.month

print("\nTEST MAE BY MONTH")
test = df[df["split"] == "test"]

monthly_mae = (
    test.groupby("month")["abs_error"]
    .agg(["mean", "count"])
)
print(monthly_mae.round(3).to_string())

print("\nTOP 10 LARGEST TEST ERRORS")
print(
    test.nlargest(10, "abs_error")[
        [
            "origin",
            "target_date",
            "horizon",
            "actual",
            "predicted",
            "abs_error",
        ]
    ].to_string(index=False)
)



MAE BY FORECAST HORIZON
split    development   test  validation
horizon                                
1              6.286  8.311       7.998
2              6.256  8.327       7.998
3              6.249  8.349       7.948
4              6.256  8.366       7.819
5              6.250  8.358       7.668
6              6.228  8.334       7.560
7              6.223  8.302       7.505

TEST MAE BY MONTH
         mean  count
month               
1       5.442    419
2       3.507    399
3       4.647    399
4       6.777    181
5      16.199    211
6      15.530    210
7      16.023    217
8      10.371    217
9      10.963    210
10     13.042    217
11      5.787    210
12      1.794    218

TOP 10 LARGEST TEST ERRORS
    origin target_date  horizon  actual  predicted  abs_error
2023-05-23  2023-05-30        7    97.7      135.0       37.3
2023-05-24  2023-05-30        6    97.7      135.0       37.3
2023-05-25  2023-05-30        5    97.7      135.0       37.3
2023-05-26  2023-05-30    

Monthly relative errors by seasons

In [22]:
df = pd.read_csv(
    DATA / "processed/seasonal_naive_predictions.csv",
    parse_dates=["origin", "target_date"],
)

test = df[df["split"] == "test"].copy()
test["month"] = test["target_date"].dt.month

monthly = test.groupby("month").agg(
    mae=("abs_error", "mean"),
    total_abs_error=("abs_error", "sum"),
    total_actual=("actual", "sum"),
    observations=("actual", "size"),
)

monthly["wape_pct"] = (
    100 * monthly["total_abs_error"]
    / monthly["total_actual"].abs()
)

print("\nMONTHLY TEST METRICS")
print(monthly.round(2).to_string())

# Analyze target dates once using the seven-day-ahead
# forecast (horizon 7), avoiding multiple predictions
# of the same target date.
h7 = test[test["horizon"] == 7].copy()

print("\nLARGEST ERRORS — UNIQUE TARGET DATES (H7)")
print(
    h7.nlargest(15, "abs_error")[
        [
            "target_date",
            "actual",
            "predicted",
            "abs_error",
        ]
    ].to_string(index=False)
)

print("\nTEST ERROR CONTRIBUTION")
total = h7["abs_error"].sum()
top10 = h7.nlargest(10, "abs_error")["abs_error"].sum()

print(
    f"Top 10 dates contribute {100 * top10 / total:.2f}% "
    "of horizon-7 absolute error"
)



MONTHLY TEST METRICS
         mae  total_abs_error  total_actual  observations  wape_pct
month                                                              
1       5.44           2280.3       35309.4           419      6.46
2       3.51           1399.3       28556.5           399      4.90
3       4.65           1854.0       27845.9           399      6.66
4       6.78           1226.6       16736.5           181      7.33
5      16.20           3418.0       21368.3           211     16.00
6      15.53           3261.3       25664.8           210     12.71
7      16.02           3476.9       27381.9           217     12.70
8      10.37           2250.5       28529.9           217      7.89
9      10.96           2302.3       25981.9           210      8.86
10     13.04           2830.1       19636.4           217     14.41
11      5.79           1215.2       14800.8           210      8.21
12      1.79            391.2       15553.7           218      2.52

LARGEST ERRORS — UNIQUE T

Trend-adjusted seasonal naive

In [23]:
OUTPUT = DATA / "processed"

df = pd.read_csv(
    OUTPUT / "delhi_energy_clean.csv",
    parse_dates=["Date"],
    index_col="Date",
).sort_index()

y = df["energy_met_MU"]

CONTEXT = 28
HORIZON = 7

records = []

for i in range(CONTEXT, len(y) - HORIZON + 1):
    history = y.iloc[i-CONTEXT:i]
    future = y.iloc[i:i+HORIZON]

    if history.isna().any() or future.isna().any():
        continue

    origin = y.index[i-1]
    target_end = future.index[-1]

    if target_end < pd.Timestamp("2022-01-01"):
        split = "development"
    elif target_end < pd.Timestamp("2023-01-01"):
        split = "validation"
    else:
        split = "test"

    values = history.to_numpy()

    # Average change between matching weekdays
    # across the last three week-over-week comparisons.
    weekly_changes = values[7:] - values[:-7]
    trend = weekly_changes.mean()

    forecast = values[-7:] + trend

    for h in range(HORIZON):
        records.append({
            "origin": origin,
            "target_date": future.index[h],
            "horizon": h + 1,
            "actual": future.iloc[h],
            "predicted": forecast[h],
            "split": split,
        })

results = pd.DataFrame(records)
results["error"] = results["actual"] - results["predicted"]
results["abs_error"] = results["error"].abs()
results["squared_error"] = results["error"] ** 2

print("\nTrend-adjusted seasonal naive baseline")

for split, part in results.groupby("split"):
    mae = part["abs_error"].mean()
    rmse = np.sqrt(part["squared_error"].mean())
    wape = (
        100 * part["abs_error"].sum()
        / part["actual"].abs().sum()
    )
    bias = part["error"].mean()

    print(
        f"{split}: "
        f"MAE={mae:.3f} MU, "
        f"RMSE={rmse:.3f}, "
        f"WAPE={wape:.2f}%, "
        f"Bias={bias:.3f} MU"
    )

results.to_csv(
    OUTPUT / "trend_naive_predictions.csv",
    index=False,
)



Trend-adjusted seasonal naive baseline
development: MAE=6.476 MU, RMSE=9.176, WAPE=7.84%, Bias=0.325 MU
test: MAE=9.181 MU, RMSE=12.721, WAPE=9.93%, Bias=0.019 MU
validation: MAE=7.952 MU, RMSE=11.257, WAPE=8.75%, Bias=0.351 MU


Build features without training

In [31]:
rows = []

for i in range(CONTEXT, len(y) - HORIZON + 1):
    history = y.iloc[i-CONTEXT:i]
    future = y.iloc[i:i+HORIZON]

    if history.isna().any() or future.isna().any():
        continue

    origin = y.index[i-1]
    values = history.to_numpy()

    base_features = {
        f"lag_{lag}": values[-lag]
        for lag in range(1, CONTEXT + 1)
    }

    base_features.update({
        "rolling_mean_7": values[-7:].mean(),
        "rolling_std_7": values[-7:].std(),
        "rolling_mean_28": values.mean(),
        "rolling_std_28": values.std(),
    })

    # Split once per complete seven-day window.
    target_end = future.index[-1]

    if target_end < pd.Timestamp("2022-01-01"):
        split = "development"
    elif target_end < pd.Timestamp("2023-01-01"):
        split = "validation"
    else:
        split = "test"

    # This loop MUST remain inside the outer loop.
    for h in range(1, HORIZON + 1):
        target_date = future.index[h-1]

        rows.append({
            **base_features,
            "origin": origin,
            "target_date": target_date,
            "horizon": h,
            "day_of_week": target_date.day_of_week,
            "month": target_date.month,
            "day_of_year": target_date.day_of_year,
            "target": future.iloc[h-1],
            "split": split,
        })

features = pd.DataFrame(rows)

print("Shape:", features.shape)
print("Duplicate keys:",
      features.duplicated(
          ["origin", "target_date", "horizon", "split"]
      ).sum())
print(features["split"].value_counts())

Shape: (15036, 40)
Duplicate keys: 0
split
development    10087
test            3108
validation      1841
Name: count, dtype: int64


In [32]:
baseline = pd.read_csv(
    DATA / "processed/seasonal_naive_predictions.csv",
    parse_dates=["origin", "target_date"],
)

keys = ["origin", "target_date", "horizon", "split"]

expected = baseline[keys + ["actual"]].rename(
    columns={"actual": "baseline_actual"}
)

aligned = features.merge(
    expected,
    on=keys,
    how="outer",
    validate="one_to_one",
    indicator=True,
)

print("Alignment results:")
print(aligned["_merge"].value_counts())

matched = aligned[aligned["_merge"] == "both"]

print(
    "Maximum target difference:",
    (matched["target"] - matched["baseline_actual"])
    .abs().max()
)

assert (aligned["_merge"] == "both").all()
assert (
    (matched["target"] - matched["baseline_actual"])
    .abs().max() < 1e-9
)

print("Baseline alignment verified.")

Alignment results:
_merge
both          15036
left_only         0
right_only        0
Name: count, dtype: int64
Maximum target difference: 0.0
Baseline alignment verified.


Check feature leakage before training - check chronological boundaries

In [33]:
for split in ["development", "validation", "test"]:
    part = features[features["split"] == split]

    print(f"\n{split.upper()}")
    print("Rows:", len(part))
    print("Forecast origins:", part["origin"].nunique())
    print("First origin:", part["origin"].min())
    print("Last origin:", part["origin"].max())
    print("First target:", part["target_date"].min())
    print("Last target:", part["target_date"].max())

dev = features[features["split"] == "development"]
val = features[features["split"] == "validation"]
test = features[features["split"] == "test"]

print("\nBoundary checks:")
print(
    "Development targets overlap validation:",
    dev["target_date"].max() >= val["target_date"].min()
)
print(
    "Validation targets overlap test:",
    val["target_date"].max() >= test["target_date"].min()
)


DEVELOPMENT
Rows: 10087
Forecast origins: 1441
First origin: 2013-04-27 00:00:00
Last origin: 2021-12-24 00:00:00
First target: 2013-04-28 00:00:00
Last target: 2021-12-31 00:00:00

VALIDATION
Rows: 1841
Forecast origins: 263
First origin: 2021-12-25 00:00:00
Last origin: 2022-11-22 00:00:00
First target: 2021-12-26 00:00:00
Last target: 2022-11-29 00:00:00

TEST
Rows: 3108
Forecast origins: 444
First origin: 2022-12-30 00:00:00
Last origin: 2024-04-21 00:00:00
First target: 2022-12-31 00:00:00
Last target: 2024-04-28 00:00:00

Boundary checks:
Development targets overlap validation: True
Validation targets overlap test: False


Remove crossing windows

In [34]:
import pandas as pd

def assigned_split(first_target, last_target):
    if last_target < pd.Timestamp("2022-01-01"):
        return "development"

    if (
        first_target >= pd.Timestamp("2022-01-01")
        and last_target < pd.Timestamp("2023-01-01")
    ):
        return "validation"

    if first_target >= pd.Timestamp("2023-01-01"):
        return "test"

    return None  # Crossing boundary

# One record per forecast origin
origin_bounds = features.groupby("origin")["target_date"].agg(
    first="min",
    last="max",
)

origin_bounds["clean_split"] = [
    assigned_split(row["first"], row["last"])
    for _, row in origin_bounds.iterrows()
]

# Exclude boundary-crossing windows
features_clean = features.drop(columns="split").merge(
    origin_bounds[["clean_split"]],
    left_on="origin",
    right_index=True,
    validate="many_to_one",
)

features_clean = features_clean.dropna(
    subset=["clean_split"]
).rename(columns={"clean_split": "split"})

print(features_clean["split"].value_counts())

for split in ["development", "validation", "test"]:
    part = features_clean[features_clean["split"] == split]
    print(
        split,
        "targets:",
        part["target_date"].min(),
        "to",
        part["target_date"].max(),
    )

assert (
    features_clean.loc[
        features_clean["split"] == "development", "target_date"
    ].max() < pd.Timestamp("2022-01-01")
)

assert (
    features_clean.loc[
        features_clean["split"] == "validation", "target_date"
    ].min() >= pd.Timestamp("2022-01-01")
)

assert (
    features_clean.loc[
        features_clean["split"] == "test", "target_date"
    ].min() >= pd.Timestamp("2023-01-01")
)

print("Chronological split boundaries verified.")

split
development    10087
test            3101
validation      1799
Name: count, dtype: int64
development targets: 2013-04-28 00:00:00 to 2021-12-31 00:00:00
validation targets: 2022-01-01 00:00:00 to 2022-11-29 00:00:00
test targets: 2023-01-01 00:00:00 to 2024-04-28 00:00:00
Chronological split boundaries verified.


Recalculate both baseline metrics

In [35]:
import numpy as np
import pandas as pd

KEYS = ["origin", "target_date", "horizon"]

def evaluate_baseline(filename, model_name):
    predictions = pd.read_csv(
        DATA / "processed" / filename,
        parse_dates=["origin", "target_date"]
    )

    # Use the corrected split assignments.
    reference = features_clean[
        KEYS + ["split", "target"]
    ]

    merged = reference.merge(
        predictions[KEYS + ["actual", "predicted"]],
        on=KEYS,
        how="left",
        validate="one_to_one"
    )

    assert merged["predicted"].notna().all()
    assert np.allclose(merged["target"], merged["actual"])

    merged["abs_error"] = (
        merged["target"] - merged["predicted"]
    ).abs()

    merged["squared_error"] = (
        merged["target"] - merged["predicted"]
    ) ** 2

    merged["error"] = (
        merged["target"] - merged["predicted"]
    )

    results = []

    for split, part in merged.groupby("split"):
        results.append({
            "model": model_name,
            "split": split,
            "mae": part["abs_error"].mean(),
            "rmse": np.sqrt(part["squared_error"].mean()),
            "wape": (
                100 * part["abs_error"].sum()
                / part["target"].abs().sum()
            ),
            "bias": part["error"].mean(),
        })

    return pd.DataFrame(results)


weekly = evaluate_baseline(
    "seasonal_naive_predictions.csv",
    "Weekly Naive"
)

trend = evaluate_baseline(
    "trend_naive_predictions.csv",
    "Trend Naive"
)

comparison = pd.concat(
    [weekly, trend],
    ignore_index=True
)

print(
    comparison.round(3).to_string(index=False)
)

       model       split   mae   rmse  wape   bias
Weekly Naive development 6.250  8.811 7.568  0.093
Weekly Naive        test 8.346 11.632 9.024  0.037
Weekly Naive  validation 7.917 10.947 8.670 -0.467
 Trend Naive development 6.476  9.176 7.842  0.325
 Trend Naive        test 9.194 12.734 9.941  0.020
 Trend Naive  validation 8.077 11.377 8.844  0.409


In [36]:
features_clean.to_parquet(
    DATA / "processed" / "gridpulse_features_v1.parquet",
    index=False
)

Prepare XGBoost inputs

In [37]:
# Notebook cell — Prepare model matrices

EXCLUDE = {
    "origin",
    "target_date",
    "target",
    "split",
}

FEATURE_COLS = [
    col for col in features_clean.columns
    if col not in EXCLUDE
]

train = features_clean[
    features_clean["split"] == "development"
].copy()

val = features_clean[
    features_clean["split"] == "validation"
].copy()

test = features_clean[
    features_clean["split"] == "test"
].copy()

X_train = train[FEATURE_COLS]
y_train = train["target"]

X_val = val[FEATURE_COLS]
y_val = val["target"]

X_test = test[FEATURE_COLS]
y_test = test["target"]

print("Features:", len(FEATURE_COLS))
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nFeature columns:")
print(FEATURE_COLS)

assert X_train.isna().sum().sum() == 0
assert X_val.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

assert set(FEATURE_COLS).isdisjoint(EXCLUDE)

print("\nXGBoost inputs prepared successfully.")

Features: 36
Training: (10087, 36)
Validation: (1799, 36)
Test: (3101, 36)

Feature columns:
['lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10', 'lag_11', 'lag_12', 'lag_13', 'lag_14', 'lag_15', 'lag_16', 'lag_17', 'lag_18', 'lag_19', 'lag_20', 'lag_21', 'lag_22', 'lag_23', 'lag_24', 'lag_25', 'lag_26', 'lag_27', 'lag_28', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_28', 'rolling_std_28', 'horizon', 'day_of_week', 'month', 'day_of_year']

XGBoost inputs prepared successfully.
